# Multi-hop Retrieval

**Multi-hop Retrieval**이란?
단일 검색으로 답을 찾을 수 없는 복합적인 질문에 대해, **단계적(Step-by-Step)으로 정보를 검색**하여 최종 답을 도출하는 기법이다.

예를 들어:
- 질문: "iPhone을 만든 회사의 본사가 위치한 도시는 어디인가?"
- 1단계(Hop 1): "iPhone을 만든 회사"를 검색 -> **Apple Inc.** 찾음
- 2단계(Hop 2): "Apple Inc.의 본사가 위치한 도시"를 검색 -> **Cupertino** 찾음
- 최종 답: **Cupertino**

In [1]:
%pip install -Uq python-dotenv langchain langchain-openai langchain-pinecone langchain-community

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = os.getenv('langsmith_key')
os.environ['LANGSMITH_PROJECT'] = 'skn23-langchain'
os.environ['OPENAI_API_KEY'] = os.getenv("openai_key")
os.environ['PINECONE_API_KEY'] = os.getenv("pinecone_key")
os.environ['COHERE_API_KEY'] = os.getenv('cohere_key')

## 데이터셋 로드

In [3]:
import pandas as pd

documents_df = pd.read_csv('./documents_multihop_v2.csv')
queries_df = pd.read_csv('./queries_multihop_v2.csv')

In [4]:
pd.set_option('display.max_colwidth', None)
documents_df

,doc_id,content
0,D1,아이폰(iPhone) 스마트폰 시리즈는 애플(Apple Inc.)에 의해 설계 및 마케팅되었습니다.
1,D2,애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.
2,D3,"쿠퍼티노(Cupertino)는 미국 캘리포니아주에 위치한 도시로, 애플의 본거지로 알려져 있습니다."
3,D4,팀 쿡(Tim Cook)은 2011년 스티브 잡스의 뒤를 이어 애플(Apple Inc.)의 CEO가 되었습니다.
4,D5,스티브 잡스(Steve Jobs)는 애플(Apple Inc.)의 공동 창업자였으며 2011년까지 CEO를 역임했습니다.
5,D6,갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.
6,D7,삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.
7,D8,이재용은 현재 삼성전자의 회장직을 맡고 있습니다.
8,D9,"서울은 대한민국의 수도이자 최대 도시로, 한강이 흐르고 있습니다."
9,D10,삼성전자는 1938년 이병철에 의해 창립된 삼성그룹의 계열사입니다.


In [5]:
queries_df

,query_id,query_text,relevant_doc_ids
0,Q1,아이폰을 만든 회사의 본사가 위치한 도시는 어디인가요?,D1=1;D2=1;D3=1
1,Q2,아이폰을 설계한 회사의 현재 CEO는 누구인가요?,D1=1;D2=1;D4=1
2,Q3,애플 본사가 위치한 도시는 어느 주에 있나요?,D2=1;D3=1
3,Q4,갤럭시 스마트폰을 만드는 회사의 현재 회장은 누구인가요?,D6=1;D7=1;D8=1
4,Q5,알파고를 개발한 회사의 본사가 위치한 도시는 어디인가요?,D14=1;D15=1
5,Q6,BTS가 소속된 기획사를 설립한 사람은 누구인가요?,D16=1;D17=1
6,Q7,영화 기생충을 연출한 감독이 수상한 영화제는 어디인가요?,D21=1;D22=1;D23=1
7,Q8,손흥민 선수가 소속된 팀의 연고지는 어디인가요?,D26=1;D27=1
8,Q9,토트넘 홋스퍼 FC가 위치한 도시가 수도인 나라는 어디인가요?,D27=1;D28=1
9,Q10,삼성전자를 창립한 사람이 세운 그룹의 이름은 무엇인가요?,D10=1


## 벡터스토어 생성 및 문서업로드

In [6]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone() # pinecone api key 인증
print(pc.list_indexes().names()) # ['winemag-data-130k-v2', 'pinecone-first']

INDEX_NAME = 'ir-multihop'
# 인덱스명 ir
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=1536,
        metric='cosine',
        spec=ServerlessSpec(
            region='us-east-1',
            cloud='aws'
        )
    )
    print(f'{INDEX_NAME}인덱스가 생성되었습니다.')
else:
    print(f'{INDEX_NAME}가 이미 존재합니다.')



['ir', 'winemag-data-130k-v2', 'pinecone-first', 'ir-meta', 'ir-compressed']


ForbiddenException: (403)
Reason: Forbidden
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'vary': 'origin, access-control-request-method, access-control-request-headers', 'access-control-allow-origin': '*', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': '34bc8d20edc51e2974a995fbb8272683', 'date': 'Mon, 09 Feb 2026 08:15:05 GMT', 'server': 'Google Frontend', 'Content-Length': '257', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"FORBIDDEN","message":"Request failed. You've reached the max serverless indexes allowed in project Default (5). Use namespaces to partition your data into logical groups, or upgrade your plan to add more serverless indexes."},"status":403}


In [ ]:
# DataFrame -> List[Document]
from langchain_core.documents import Document

documents = []
for idx, row in documents_df.iterrows():
    doc_id = row['doc_id']
    content = row['content']
    doc = Document(
        page_content=content,
        metadata={
            'doc_id': doc_id
        }
    )
    documents.append(doc)

print(len(documents))

30


In [ ]:
# 벡터스토어객체 생성 및 업로드
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

embedding_model = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = PineconeVectorStore(
    index_name=INDEX_NAME,
    embedding=embedding_model
) # 이미 존재하는 인덱스에 연결만
vector_store.add_documents(documents) # 문서업로드

# vector_store = PineconeVectorStore.from_documents() # 이미 존재하는 인덱스에 연결 & 문서업서트

['e057eaef-7e6f-4d87-a7e0-85b8eb6e9676',
 '6d4ce93c-fa6d-49b1-9a72-1d4670b764a4',
 'f93f6bf3-7bc4-4b55-b213-6afbb2b3c351',
 '29b9e398-c3d9-42d0-9b32-78235738abc1',
 '37e6b751-a12e-4e69-82d0-e1962c8cde78',
 '2039fa62-9063-4774-bb42-474cdb73324b',
 '6ec64310-471e-4e46-8cfc-64ef82045a52',
 '17f64338-5d9b-4b9b-b745-49952473ba21',
 'c9b7bb1f-4ff2-4459-b427-dd99d0371862',
 '1d8dbbad-49e1-46ea-a2f4-fc24222927ff',
 '303a63e1-ec36-49c7-a6ef-896cdf4ce610',
 'e4e66894-2fb4-410f-98de-8db93bc420fa',
 'd8dd1587-c947-4568-ab19-475c75e6c056',
 '171bb92b-eac9-4937-b654-5103595acbb7',
 '78429c67-36f6-4bf6-87e8-3c1b9888e6ba',
 '44570598-2af0-4fbd-aa68-b7e47b10662e',
 'bdb810d2-46dd-4e48-98f4-cd85e9589942',
 'e933c5c2-3e4a-48f9-92cb-1b6912120592',
 'a41064a6-5736-48b1-8f66-d869c41d2c60',
 '7b5b708c-b8ef-4efa-9339-ed227569ec15',
 '0f4c1475-b981-4807-9094-61eb6cf68915',
 '74602755-5a60-48f5-a7cd-e69e4e9dbe88',
 'be70f03f-579b-45b7-9680-88dfa8c29c2d',
 '325c9509-9d78-4a94-bc13-1aca7502f51b',
 '9447f40a-7713-

In [ ]:
# 벡터 서치
vector_store.similarity_search('손흥민이 뭐하는 사람이야?')

[Document(id='f75959d4-4de8-4c62-b199-fe5dc105fbbb', metadata={'doc_id': 'D29'}, page_content='손흥민은 2021-2022 시즌 프리미어리그에서 아시아 선수 최초로 득점왕(골든 부트)을 수상했습니다.'),
 Document(id='7d462dc4-9e80-410d-ad17-c4cb01af145a', metadata={'doc_id': 'D26'}, page_content='손흥민은 잉글랜드 프리미어리그(EPL)의 토트넘 홋스퍼 FC에서 활약하는 대한민국 축구 선수입니다.'),
 Document(id='58a61a1c-d67e-4e06-abd3-ad553915dae6', metadata={'doc_id': 'D30'}, page_content='해리 케인은 토트넘에서 손흥민과 환상적인 호흡을 보여준 잉글랜드 출신 스트라이커입니다.'),
 Document(id='6ec64310-471e-4e46-8cfc-64ef82045a52', metadata={'doc_id': 'D7'}, page_content='삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.')]

## Multihop RAG 구현

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-4.1-mini', temperature=0)
prompt = PromptTemplate.from_template('''
당신은 복잡한 질문에 답하기 위해 정보를 단계적으로 검색하는 AI에이젼트입니다.
사용자의 질문과 현재까지 수집된 정보를 바탕으로,
1. 아직 답을 찾지 못했다면:
  다음에 검색해야 할 가장 구체적이고 필요한 검색어(Query)를 출력하세요
2. 충분한 정보를 찾았다면:
  'ANSWER: '뒤에 최종 정답을 적어서 출력하세요.

### 사용자의 원래질문 ###
{original_question}

### 현재까지 수집된 정보 Context ###
{context}

### 출력지시사항 ###
불필요한 설명없이, '검색어' 또는 'ANSWER: 정답' 형식으로만 출력하세요.

1.추가검색이 필요한 경우, 검색어는 "BTS가 소속된 기획사"인 경우
("검색어" 출력하지 말것)
출력: BTS가 소속된 기획사

2.정답 추론이 가능한 경우
출력: ANSWER: 하이브
''')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

print(chain.invoke({'original_question': 'BTS가 소속된 기획사를 설립한 사람은 누구인가요?', 'context': ''}))
print(chain.invoke({'original_question': 'BTS가 소속된 기획사를 설립한 사람은 누구인가요?', 'context': '방시혁이 BTS의 기획사 하이브를 만들었다.'}))

BTS가 소속된 기획사 설립자
ANSWER: 방시혁


In [ ]:
# 멀티홉 검색을 통해 단계적으로 정답을 도출하는 함수
def multihop_search(question, max_hop=3):
    context = '아직 수집된 정보 없음'
    retrieved_doc_ids = set() # 중복제거

    print(f'질문: {question}')

    for i in range(max_hop):
        print(f'\n----- {i + 1} hop -----')

        # 1.에이젼트 질의
        response = chain.invoke({'original_question': question, 'context': context})

        # 2.정답도출 여부 확인
        if response.startswith('ANSWER:'):
            final_answer = response.replace('ANSWER:', '').strip()

            print(f'\n정답: {final_answer}')
            return final_answer, retrieved_doc_ids

        # 3.검색
        query = response
        docs = vector_store.similarity_search(query, k=3)
        # 검색문서 관리
        for doc in docs:
            retrieved_doc_ids.add(doc.metadata['doc_id'])

        # 수집된 정보를 하나의 텍스트로 병합
        content = '\n'.join([doc.page_content for doc in docs])
        print(f'\n검색어: {query}')
        print(f'검색결과:\n{content}')
        if context == '아직 수집된 정보 없음':
            context = content
        else:
            context += '\n\n' + content

    print('😥제한된 hop 수 내에 정답을 찾지 못했습니다.😥')

    return '😥답변실패😥', retrieved_doc_ids

In [ ]:
answer, retrieved_doc_ids = multihop_search('BTS가 소속된 기획사를 설립한 사람은?')
print(answer)
print(retrieved_doc_ids)

질문: BTS가 소속된 기획사를 설립한 사람은?

----- 1 hop -----

검색어: BTS가 소속된 기획사 설립자
검색결과:
하이브(HYBE)는 작곡가 겸 프로듀서 방시혁이 설립한 엔터테인먼트 기업입니다.
BTS(방탄소년단)는 2013년 데뷔한 대한민국 7인조 보이그룹으로, 하이브(HYBE) 소속입니다.
BTS의 영어 곡 'Dynamite'는 한국 가수 최초로 미국 빌보드 핫 100 차트 1위를 기록했습니다.

----- 2 hop -----

정답: 방시혁
방시혁
{'D17', 'D16', 'D20'}


In [ ]:
answer, retrieved_doc_ids = multihop_search('영화 기생충이 상을 받은 영화제는?')
print(answer)
print(retrieved_doc_ids)

질문: 영화 기생충이 상을 받은 영화제는?

----- 1 hop -----

검색어: 영화 기생충이 수상한 영화제 목록
검색결과:
봉준호 감독은 영화 '기생충'으로 칸 영화제에서 최고상인 황금종려상을 수상했습니다.
'기생충'은 제92회 아카데미 시상식(오스카)에서 작품상, 감독상 등 4관왕을 달성했습니다.
영화 '기생충(Parasite)'은 2019년 개봉한 봉준호 감독의 블랙 코미디 스릴러 영화입니다.

----- 2 hop -----

정답: 칸 영화제 황금종려상, 제92회 아카데미 시상식 작품상·감독상 포함 4관왕
칸 영화제 황금종려상, 제92회 아카데미 시상식 작품상·감독상 포함 4관왕
{'D21', 'D24', 'D22'}


In [ ]:
answer, retrieved_doc_ids = multihop_search('아이폰 만든 회사의 본사는 어느 주에 있는가?')
print(answer)
print(retrieved_doc_ids)

질문: 아이폰 만든 회사의 본사는 어느 주에 있는가?

----- 1 hop -----

검색어: 애플 본사 위치
검색결과:
애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.
쿠퍼티노(Cupertino)는 미국 캘리포니아주에 위치한 도시로, 애플의 본거지로 알려져 있습니다.
하이브의 본사는 대한민국 서울 용산구에 위치한 하이브 용산 사옥입니다.

----- 2 hop -----

정답: 캘리포니아
캘리포니아
{'D19', 'D3', 'D2'}


In [ ]:
answer, retrieved_doc_ids = multihop_search('한국에 들어오는 테슬라 차량을 생산하는 곳은?')
print(answer)
print(retrieved_doc_ids)

질문: 한국에 들어오는 테슬라 차량을 생산하는 곳은?

----- 1 hop -----

검색어: 테슬라 한국 수입 차량 생산지
검색결과:
갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.
손흥민은 잉글랜드 프리미어리그(EPL)의 토트넘 홋스퍼 FC에서 활약하는 대한민국 축구 선수입니다.
서울은 대한민국의 수도이자 최대 도시로, 한강이 흐르고 있습니다.

----- 2 hop -----

검색어: 테슬라 차량 생산 공장 한국
검색결과:
삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.
애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.
갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.

----- 3 hop -----

검색어: 테슬라 한국 차량 생산지
검색결과:
갤럭시(Galaxy) 시리즈는 삼성전자가 제조 및 판매하는 안드로이드 스마트폰 브랜드입니다.
삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.
애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.
😥제한된 hop 수 내에 정답을 찾지 못했습니다.😥
😥답변실패😥
{'D6', 'D7', 'D26', 'D2', 'D9'}


## 검색 성능평가

In [ ]:
def parse_relevant_docs(relevant_str):
    """D1=1;D2=1 형태의 문자열을 파싱해서 문서ID집합을 반환"""
    return {item.split('=')[0] for item in relevant_str.split(';')}
parse_relevant_docs('D1=1;D2=1')

{'D1', 'D2'}

In [ ]:
from tqdm.auto import tqdm

def evaluate_multihop():
    results = []

    for _, row in tqdm(queries_df.iterrows(), total=len(queries_df)):
        qid = row['query_id']
        query_text = row['query_text']
        relevant_docs = parse_relevant_docs(row['relevant_doc_ids'])

        # multihop수행
        answer, retrieved_docs = multihop_search(query_text)
        print('-' * 50, end='\n\n')

        # 평가 메트릭계산
        # recall = 실제 찾은 문서수 / 정답 문서수
        intersection = relevant_docs.intersection(retrieved_docs)
        recall = len(intersection) / len(relevant_docs)

        results.append({
            'qid': qid,
            'query_text': query_text,
            'answer': answer,
            'relevant_docs': relevant_docs,
            'retrieved_docs': retrieved_docs,
            'recall': recall
        })
    return results

results = evaluate_multihop()

  0%|          | 0/10 [00:00<?, ?it/s]

질문: 아이폰을 만든 회사의 본사가 위치한 도시는 어디인가요?

----- 1 hop -----

검색어: 아이폰을 만든 회사 본사 위치
검색결과:
애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.
삼성전자의 본사는 대한민국 서울 서초구에 위치해 있습니다.
아이폰(iPhone) 스마트폰 시리즈는 애플(Apple Inc.)에 의해 설계 및 마케팅되었습니다.

----- 2 hop -----

정답: 캘리포니아 쿠퍼티노
--------------------------------------------------

질문: 아이폰을 설계한 회사의 현재 CEO는 누구인가요?

----- 1 hop -----

검색어: 애플 현재 CEO
검색결과:
스티브 잡스(Steve Jobs)는 애플(Apple Inc.)의 공동 창업자였으며 2011년까지 CEO를 역임했습니다.
팀 쿡(Tim Cook)은 2011년 스티브 잡스의 뒤를 이어 애플(Apple Inc.)의 CEO가 되었습니다.
애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.

----- 2 hop -----

검색어: 애플 현재 CEO
검색결과:
스티브 잡스(Steve Jobs)는 애플(Apple Inc.)의 공동 창업자였으며 2011년까지 CEO를 역임했습니다.
팀 쿡(Tim Cook)은 2011년 스티브 잡스의 뒤를 이어 애플(Apple Inc.)의 CEO가 되었습니다.
애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국적 기술 회사입니다.

----- 3 hop -----

검색어: 애플 현재 CEO
검색결과:
스티브 잡스(Steve Jobs)는 애플(Apple Inc.)의 공동 창업자였으며 2011년까지 CEO를 역임했습니다.
팀 쿡(Tim Cook)은 2011년 스티브 잡스의 뒤를 이어 애플(Apple Inc.)의 CEO가 되었습니다.
애플(Apple Inc.)은 캘리포니아 쿠퍼티노에 본사를 둔 미국의 다국

In [ ]:
results_df = pd.DataFrame(results)
print(f'최종 recall : {results_df['recall'].mean():.4f}')

results_df

최종 recall : 0.7833


,qid,query_text,answer,relevant_docs,retrieved_docs,recall
0,Q1,아이폰을 만든 회사의 본사가 위치한 도시는 어디인가요?,캘리포니아 쿠퍼티노,"{D1, D3, D2}","{D1, D7, D2}",0.666667
1,Q2,아이폰을 설계한 회사의 현재 CEO는 누구인가요?,😥답변실패😥,"{D1, D4, D2}","{D4, D5, D2}",0.666667
2,Q3,애플 본사가 위치한 도시는 어느 주에 있나요?,캘리포니아,"{D3, D2}","{D13, D3, D2}",1.000000
3,Q4,갤럭시 스마트폰을 만드는 회사의 현재 회장은 누구인가요?,😥답변실패😥,"{D7, D6, D8}","{D10, D7, D8}",0.666667
4,Q5,알파고를 개발한 회사의 본사가 위치한 도시는 어디인가요?,영국 런던,"{D14, D15}","{D14, D13, D7, D15, D12, D2}",1.000000
5,Q6,BTS가 소속된 기획사를 설립한 사람은 누구인가요?,방시혁,"{D17, D16}","{D17, D16, D20}",1.000000
6,Q7,영화 기생충을 연출한 감독이 수상한 영화제는 어디인가요?,"칸 영화제, 아카데미 시상식(오스카)","{D21, D23, D22}","{D25, D24, D22}",0.333333
7,Q8,손흥민 선수가 소속된 팀의 연고지는 어디인가요?,영국 런던,"{D27, D26}","{D27, D26, D29}",1.000000
8,Q9,토트넘 홋스퍼 FC가 위치한 도시가 수도인 나라는 어디인가요?,영국,"{D27, D28}","{D27, D26, D9}",0.500000
9,Q10,삼성전자를 창립한 사람이 세운 그룹의 이름은 무엇인가요?,삼성그룹,{D10},"{D10, D7, D8}",1.000000


### 핵심 결과
- 멀티홉 검색은 **단일 검색으로는 해결하기 어려운 복합 질의**에서 효과적으로 동작했다.
- 질문을 단계적으로 분해하고, 각 단계에서 필요한 정보를 순차적으로 수집함으로써  
  **추론 기반 질의에 대한 Recall을 안정적으로 확보**할 수 있었다.

---

### 왜 멀티홉이 필요한가
- 단일 Dense Retrieval은 **한 문서 안에 모든 단서가 존재한다는 가정**에 의존한다.
- 실제 질의는  
  - *“A와 관련된 B는 무엇인가?”*  
  - *“A를 만든 회사의 본사는 어디인가?”*  
  와 같이 **여러 문서에 정보가 분산**된 경우가 많다.
- 멀티홉 검색은 이 문제를  
  **검색 → 추론 → 추가 검색**의 반복 구조로 해결한다.

---

### 실험을 통해 확인된 장점
- 단계별 검색으로 **정답 문서 회수율(Recall) 향상**
- 검색 과정에서 실제로 참조된 문서를 추적 가능
- LLM이 “지금 무엇을 더 찾아야 하는지”를 스스로 판단하여  
  **질의 전개(Query Decomposition)**가 자연스럽게 수행됨

---

### 한계점
- 홉(hop) 수 증가에 따라 **지연 시간과 비용 증가**
- 중간 검색이 잘못되면 이후 단계도 함께 실패할 가능성 존재
- Precision보다는 **Recall 중심 평가에 더 적합**

### 실무 적용 결론
단일 검색으로 해결 가능 → 일반 Retrieval  
복합 추론이 필요한 질문 → Multi-hop Retrieval


- 멀티홉 검색은 **지식 탐색형 QA, 리서치, 에이전트 기반 RAG**에 특히 적합
- 실제 서비스에서는  
  **Self-Query / Metadata → Dense → Multi-hop → ReRank**  
  형태로 결합하는 것이 가장 현실적인 전략이다.

---

### 한 줄 결론 (교안 / 발표용)
> **멀티홉 검색은 분산된 정보를 단계적으로 연결하여, 복합 질의에 대한 추론 가능성을 확장하는 검색 전략이다.**